# nnUNet Preprocessing for Vesuvius Surface Detection

**Based on:** surface-nnunet-training-inference-with-2xt4.ipynb

**Purpose:** Preprocess data for 3d_fullres configuration (Host Baseline)

**Output:** `/kaggle/working/nnUNet_preprocessed/` for use as Kaggle dataset

## 1. Configuration & Imports

**Section Summary:**
This section defines all configurable parameters for the pipeline. Modify these
to adapt the baseline to your needs.

**Key Configuration Decisions:**
- `FOLD="all"`: Trains on all data without cross-validation (faster for submissions)
- `CONFIGURATION="3d_lowres"`: Good balance of speed and quality for large volumes
- `NUM_GPUS=auto`: Uses all available GPUs for DDP training
- `EPOCHS=None`: Default 1000 epochs (can reduce to 250-500 if converged)

**Preprocessed Data Caching:**
To skip the 1-2 hour preprocessing step:
1. Run preprocessing once
2. Upload `nnUNet_preprocessed` folder as Kaggle dataset
3. Set `PREPARED_PREPROCESSED_PATH` to that dataset path

**Path Structure:**
- INPUT_DIR: Competition data (read-only)
- WORKING_DIR: Temporary files (/kaggle/temp, cleared between sessions)
- OUTPUT_DIR: Persistent outputs (/kaggle/working)

In [ ]:
import os
import json
import shutil
import subprocess
from functools import partial
from multiprocessing import Pool
from pathlib import Path
from typing import Optional, Tuple, List, Literal, Union

# =============================================================================
# TYPE DEFINITIONS
# =============================================================================

# Available epoch counts from nnUNet's built-in trainers
# These are pre-defined trainer classes in nnUNet - you cannot use arbitrary values
# See: nnunetv2/training/nnUNetTrainer/variants/training_length/nnUNetTrainer_Xepochs.py
Epochs = Literal[1, 5, 10, 20, 50, 100, 250, 500, 750, 1000, 2000, 4000, 8000]

# =============================================================================
# CONFIGURATION - MODIFY THESE FOR YOUR SETUP
# =============================================================================

# Competition data path (Kaggle dataset)
INPUT_DIR = Path("/kaggle/input/vesuvius-challenge-surface-detection")

# Pre-prepared nnUNet preprocessed dataset
# Upload your preprocessed data as a Kaggle dataset to skip the 1-2 hour preprocessing step
# Set to non-existent path to force fresh preprocessing
PREPARED_PREPROCESSED_PATH = Path("/kaggle/input/vesuvius-surface-nnunet-preprocessed")

# Working directories
WORKING_DIR = Path("/kaggle/temp")  # Large intermediate files (cleared between sessions)
OUTPUT_DIR = Path("/kaggle/working")  # Final outputs (persisted)

# nnUNet directory structure (follows nnUNet conventions)
NNUNET_BASE = WORKING_DIR / "nnUNet_data"
NNUNET_RAW = NNUNET_BASE / "nnUNet_raw"  # Small - uses symlinks to competition data
NNUNET_PREPROCESSED = OUTPUT_DIR / "nnUNet_preprocessed"  # Changed: output to /kaggle/working for dataset export  # Large - can use pre-prepared dataset
NNUNET_RESULTS = OUTPUT_DIR / "nnUNet_results"  # Trained models go here (persisted)

# Dataset configuration
DATASET_ID = 100  # nnUNet dataset ID (arbitrary, just needs to be consistent)
DATASET_NAME = f"Dataset{DATASET_ID:03d}_VesuviusSurface"

# =============================================================================
# TRAINING CONFIGURATION
# =============================================================================

# Fold: 0-4 for 5-fold cross-validation, "all" for training on all data
# - Use "all" for final submission (faster, uses all training data)
# - Use 0-4 for model validation and ensembling
FOLD: Union[int, str] = "all"

# Configuration: determines network architecture and resolution
# - "2d": Fastest, processes slice-by-slice (good for quick experiments)
# - "3d_lowres": Fast, handles large volumes well (NOTE: creates predicted_next_stage for cascade)
# - "3d_fullres": Best quality, single-stage (RECOMMENDED)
# - "3d_cascade_fullres": Two-stage for very large volumes (slowest, uses 3d_lowres predictions)
CONFIGURATION = "3d_fullres"  # Changed: Host Baseline uses 3d_fullres

# Planner: determines network architecture variant
# - "nnUNetPlanner": Standard U-Net encoder
# - "nnUNetPlannerResEncM": ResNet encoder, medium size (RECOMMENDED - often 1-2% better)
# - "nnUNetPlannerResEncL": ResNet encoder, large (more parameters, slower)
PLANNER = "nnUNetPlannerResEncM"
PLANS_NAME = "nnUNetResEncUNetMPlans"  # Must match planner (see reference table in header)

# Number of CPU workers for preprocessing and data preparation
NUM_WORKERS = os.cpu_count() or 4

# Epochs: Number of training epochs
# - None or 1000: Default full training
# - 50-100: Quick experiments
# - 250-500: Good balance (check progress.png to see if converged)
# Must be one of: 1, 5, 10, 20, 50, 100, 250, 500, 750, 1000, 2000, 4000, 8000
EPOCHS: Optional[Epochs] = None

# =============================================================================
# HOST BASELINE CONFIGURATION
# =============================================================================
# Custom trainer for Host Baseline (0.543 LB raw, 0.562 w/ post-processing)
# Set TRAINER to use custom nnUNetTrainerMedialSurfaceRecall
# Set CUSTOM_PLANS to use custom architecture configuration

# Trainer: Custom trainer class name
# - "nnUNetTrainer": Standard nnUNet trainer (default)
# - "nnUNetTrainerMedialSurfaceRecall": Host Baseline with Skeleton Recall Loss
TRAINER: str = "nnUNetTrainer"

# Custom Plans: Use custom architecture plans for Host Baseline
# - None: Use standard plans from PLANS_NAME
# - "nnUNetHostBaselinePlans": Custom plans with deeper ResidualEncoderUNet
CUSTOM_PLANS: Optional[str] = None

# Command timeout in seconds (None for no timeout)
# Useful for Kaggle's 9-hour limit - set to e.g., 28800 (8 hours) to leave buffer
COMMAND_TIMEOUT: Optional[int] = None


def _get_gpu_count() -> int:
    """Get number of available CUDA GPUs."""
    try:
        import torch
        return torch.cuda.device_count() if torch.cuda.is_available() else 1
    except ImportError:
        return 0


# Number of GPUs for DDP training
# Auto-detected by default. Set to 1 to disable multi-GPU.
# NOTE: Multi-GPU DDP can sometimes hang in notebook environments.
# If training hangs, try num_gpus=1
NUM_GPUS: int = _get_gpu_count()


# =============================================================================
# PATH HELPER FUNCTIONS
# =============================================================================

def _get_trainer_name_simple(epochs: Optional[int], trainer: str = "nnUNetTrainer") -> str:
    """Get trainer class name based on epochs (simple version for path construction)."""
    # For custom trainers, return as-is (they handle epochs differently)
    if trainer != "nnUNetTrainer":
        return trainer
    if epochs is None or epochs == 1000:
        return "nnUNetTrainer"
    elif epochs == 1:
        return "nnUNetTrainer_1epoch"  # Special case: singular form
    else:
        return f"nnUNetTrainer_{epochs}epochs"


def get_training_output_dir(
    epochs: Optional[Epochs] = None,
    plans: str = PLANS_NAME,
    config: str = CONFIGURATION,
    fold: Union[int, str] = FOLD,
    trainer: str = TRAINER
) -> Path:
    """
    Get the training output directory path based on configuration.
    
    nnUNet creates this folder structure:
    NNUNET_RESULTS/Dataset100_VesuviusSurface/nnUNetTrainer__nnUNetResEncUNetMPlans__3d_lowres/fold_all/
    
    Use this to find checkpoints, logs, and progress.png
    """
    _epochs = epochs if epochs is not None else EPOCHS
    trainer_name = _get_trainer_name_simple(_epochs, trainer)
    return NNUNET_RESULTS / DATASET_NAME / f"{trainer_name}__{plans}__{config}" / f"fold_{fold}"


def get_progress_image_path(
    epochs: Optional[Epochs] = None,
    plans: str = PLANS_NAME,
    config: str = CONFIGURATION,
    fold: Union[int, str] = FOLD,
    trainer: str = TRAINER
) -> Path:
    """Get path to training progress image (loss curves, metrics over epochs)."""
    return get_training_output_dir(epochs, plans, config, fold, trainer) / "progress.png"

## 2. Environment Setup

**Section Summary:**
Sets up nnUNet environment variables and directory structure.

**nnUNet Environment Variables:**
- `nnUNet_raw`: Where nnUNet looks for raw dataset
- `nnUNet_preprocessed`: Where preprocessed data is stored
- `nnUNet_results`: Where trained models are saved
- `nnUNet_compile`: Disable torch.compile (can cause issues)

**Pre-prepared Data Handling:**
The `_link_prepared_preprocessed()` function handles cached preprocessed data:
- Copies metadata files (JSON, PKL) that nnUNet may need to modify
- Symlinks heavy data files (NPZ, B2ND) to save space
- Handles both direct dataset paths and parent folder paths

In [ ]:
def setup_environment():
    """Set up nnUNet environment variables and directories."""
    for d in [NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS, OUTPUT_DIR]:
        d.mkdir(parents=True, exist_ok=True)
    
    os.environ["nnUNet_raw"] = str(NNUNET_RAW)
    os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
    os.environ["nnUNet_results"] = str(NNUNET_RESULTS)
    os.environ["nnUNet_compile"] = "true"
    
    print(f"nnUNet_raw: {NNUNET_RAW}")
    print(f"nnUNet_preprocessed: {NNUNET_PREPROCESSED}")
    print(f"nnUNet_results: {NNUNET_RESULTS}")
    print(f"nnUNet_USE_BLOSC2: {os.environ.get('nnUNet_USE_BLOSC2', 'not set')} (0=NPZ, 1=blosc2)")
    print(f"NUM_WORKERS: {NUM_WORKERS}")


def _link_prepared_preprocessed() -> bool:
    """
    Link/copy pre-prepared preprocessed data if available.
    
    Copies the folder structure and metadata files (which nnUNet may need to modify),
    but symlinks the heavy .npz/.b2nd data files to save space.
    
    Handles two possible structures:
    1. PREPARED_PREPROCESSED_PATH points directly to Dataset100_* folder
    2. PREPARED_PREPROCESSED_PATH contains Dataset100_* as subfolder
    
    Returns True if linked/copied successfully.
    """
    if not PREPARED_PREPROCESSED_PATH.exists():
        return False
    
    # Determine source directory - could be the path itself or a subfolder
    source_dir = PREPARED_PREPROCESSED_PATH
    if not (source_dir / "dataset.json").exists():
        # Look for Dataset folder inside
        dataset_folders = list(PREPARED_PREPROCESSED_PATH.glob(f"Dataset*_{DATASET_NAME.split('_')[1]}*"))
        if not dataset_folders:
            dataset_folders = list(PREPARED_PREPROCESSED_PATH.glob("Dataset*"))
        if dataset_folders:
            source_dir = dataset_folders[0]
        else:
            print(f"No dataset folder found in {PREPARED_PREPROCESSED_PATH}")
            return False
    
    target_dir = NNUNET_PREPROCESSED / DATASET_NAME
    
    if target_dir.exists():
        print(f"Preprocessed data already exists: {target_dir}")
        return True
    
    print(f"Linking preprocessed data from: {source_dir}")
    target_dir.mkdir(parents=True, exist_ok=True)
    
    # Files that nnUNet may need to write/modify - copy these
    copy_patterns = ['*.json', '*.pkl', '*.txt']
    
    # Heavy data files - symlink these  
    symlink_patterns = ['*.npz', '*.npy', '*.b2nd']
    
    copied = 0
    linked = 0
    
    for src_path in source_dir.rglob('*'):
        if src_path.is_dir():
            continue
            
        # Compute relative path and create target path
        rel_path = src_path.relative_to(source_dir)
        dst_path = target_dir / rel_path
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Check if this is a heavy data file (symlink) or metadata (copy)
        is_data_file = any(src_path.match(pat) for pat in symlink_patterns)
        
        if is_data_file:
            if not dst_path.exists():
                dst_path.symlink_to(src_path.resolve())
                linked += 1
        else:
            if not dst_path.exists():
                shutil.copy2(src_path, dst_path)
                copied += 1
    
    print(f"Prepared preprocessed data: {copied} files copied, {linked} files symlinked")
    print(f"Location: {target_dir}")
    return True


setup_environment()

## 3. Installation & Imports

**Section Summary:**
Installs required packages and imports libraries.

**Key Dependencies:**
- `nnunetv2`: The nnUNet framework (includes PyTorch)
- `nibabel`: For loading NIfTI files (nnUNet's default format)
- `tifffile`: For reading/writing TIFF files (competition format)
- `matplotlib`: For visualization

**Important Notes:**
- Set `nnUNet_USE_BLOSC2=1` BEFORE importing nnunetv2 to use blosc2 format (faster)
- Blosc2 format can cause issues on some systems

In [ ]:
!mkdir -p /kaggle/temp
!mkdir -p predictions_tiff

# Install nnUNet and dependencies
# Note: scikit-image is needed for Host Baseline (skeletonization, Frangi filter)
!pip install nnunetv2 nibabel tifffile tqdm -q --no-index -f "/kaggle/input/surface-packages-offline"

# Kaggle has scipy/scikit-image pre-installed, but verify they're importable
try:
    from skimage.morphology import skeletonize
    from skimage.filters import frangi
    from scipy import ndimage
    print("scikit-image and scipy: OK")
except ImportError as e:
    print(f"Missing dependency: {e}")
    print("Installing scikit-image...")
    import subprocess
    subprocess.run(["pip", "install", "scikit-image", "-q"])

# IMPORTANT: Set this BEFORE importing nnunetv2
# Blosc2 is a newer compression format but can cause compatibility issues
os.environ["nnUNet_USE_BLOSC2"] = "1"  # Use blosc2 format (faster, smaller files)

import nibabel as nib
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Show GPU configuration
print(f"Available GPUs: {_get_gpu_count()}")
print(f"Using NUM_GPUS={NUM_GPUS}")

## 3.5 Custom Trainer Installation (Host Baseline)

**Section Summary:**
Installs custom nnUNet components for Host Baseline implementation.

**Components:**
1. **SoftSkeletonRecallLoss**: Topology-preserving loss based on skeletonization
2. **DC_SkelREC_and_CE_loss**: Combined Dice + CE + Skeleton Recall loss
3. **MedialSurfaceTransform**: 2D slice-based skeletonization for surfaces
4. **nnUNetTrainerMedialSurfaceRecall**: Custom trainer using the above

**Host Baseline Specifications:**
- Architecture: ResidualEncoderUNet with deeper blocks [1, 3, 4, 6, 6, 6]
- Patch size: [192, 192, 192]
- Loss: Dice + CE + Skeleton Recall (weights 1.0, 1.0, 1.0)
- Post-processing: threshold 0.75 + Frangi surfaceness filter

**Usage:**
```python
# Install custom trainer before training
install_custom_trainer()

# Then train with custom trainer
full_pipeline(trainer="nnUNetTrainerMedialSurfaceRecall", custom_plans="nnUNetHostBaselinePlans")
```

In [ ]:
# =============================================================================
# CUSTOM TRAINER FOR HOST BASELINE
# =============================================================================
# This section installs custom nnUNet components for the Host Baseline:
# - SoftSkeletonRecallLoss: Topology-preserving loss based on skeletonization
# - DC_SkelREC_and_CE_loss: Combined Dice + CE + Skeleton Recall loss
# - MedialSurfaceTransform: 2D slice-based skeletonization transform
# - nnUNetTrainerMedialSurfaceRecall: Custom trainer using above components

def install_custom_trainer():
    """
    Install custom trainer files into nnunetv2 package directory.
    
    This function writes the necessary Python files for the Host Baseline
    custom trainer directly into the nnunetv2 installation directory.
    
    Components installed:
    1. skeleton_recall.py - SoftSkeletonRecallLoss
    2. compound_losses.py - DC_SkelREC_and_CE_loss  
    3. medial_surface.py - MedialSurfaceTransform
    4. nnUNetTrainerMedialSurfaceRecall.py - Custom trainer
    5. nnUNetHostBaselinePlans.json - Custom architecture plans
    
    Returns:
        Path to nnunetv2 package directory
    """
    import nnunetv2
    nnunet_path = Path(nnunetv2.__file__).parent
    
    print(f"Installing custom trainer to: {nnunet_path}")
    
    # ==========================================================================
    # 1. Create loss directory and write skeleton_recall.py
    # ==========================================================================
    loss_dir = nnunet_path / "training" / "loss"
    loss_dir.mkdir(parents=True, exist_ok=True)
    
    skeleton_recall_code = '''"""
Soft Skeleton Recall Loss for topology-preserving segmentation.

Based on the clDice paper and official implementations.
Computes skeleton recall by comparing predicted skeleton with ground truth skeleton.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Callable


def soft_erode(img: torch.Tensor) -> torch.Tensor:
    """Soft morphological erosion using min pooling."""
    if len(img.shape) == 4:
        # 2D: (B, C, H, W)
        p1 = -F.max_pool2d(-img, kernel_size=(3, 1), stride=(1, 1), padding=(1, 0))
        p2 = -F.max_pool2d(-img, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1))
        return torch.min(p1, p2)
    elif len(img.shape) == 5:
        # 3D: (B, C, D, H, W)
        p1 = -F.max_pool3d(-img, kernel_size=(3, 1, 1), stride=(1, 1, 1), padding=(1, 0, 0))
        p2 = -F.max_pool3d(-img, kernel_size=(1, 3, 1), stride=(1, 1, 1), padding=(0, 1, 0))
        p3 = -F.max_pool3d(-img, kernel_size=(1, 1, 3), stride=(1, 1, 1), padding=(0, 0, 1))
        return torch.min(torch.min(p1, p2), p3)
    else:
        raise ValueError(f"Expected 4D or 5D input, got {len(img.shape)}D")


def soft_dilate(img: torch.Tensor) -> torch.Tensor:
    """Soft morphological dilation using max pooling."""
    if len(img.shape) == 4:
        # 2D
        return F.max_pool2d(img, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    elif len(img.shape) == 5:
        # 3D
        return F.max_pool3d(img, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    else:
        raise ValueError(f"Expected 4D or 5D input, got {len(img.shape)}D")


def soft_open(img: torch.Tensor) -> torch.Tensor:
    """Soft morphological opening (erosion followed by dilation)."""
    return soft_dilate(soft_erode(img))


def soft_skel(img: torch.Tensor, num_iter: int = 10) -> torch.Tensor:
    """
    Compute soft skeleton using iterative morphological operations.
    
    This approximates skeletonization by iteratively eroding the image
    and subtracting the opening at each scale.
    
    Args:
        img: Input tensor (B, C, ...) with values in [0, 1]
        num_iter: Number of iterations (skeleton thickness)
    
    Returns:
        Soft skeleton tensor
    """
    img_eroded = img.clone()
    skel = F.relu(img - soft_open(img))
    
    for _ in range(num_iter):
        img_eroded = soft_erode(img_eroded)
        delta = F.relu(img_eroded - soft_open(img_eroded))
        skel = skel + F.relu(delta - skel * delta)
    
    return skel


class SoftSkeletonRecallLoss(nn.Module):
    """
    Soft Skeleton Recall Loss.
    
    Computes the recall of the ground truth skeleton in the predicted segmentation.
    This encourages the model to preserve topological structures (thin connections).
    
    Loss = 1 - (sum(pred * gt_skel) + smooth) / (sum(gt_skel) + smooth)
    
    Args:
        smooth: Smoothing factor to avoid division by zero
        num_iter: Number of iterations for soft skeletonization
        do_bg: Whether to include background class (default False)
        apply_nonlin: Optional nonlinearity to apply to predictions (e.g., softmax)
    """
    
    def __init__(
        self,
        smooth: float = 1.0,
        num_iter: int = 10,
        do_bg: bool = False,
        apply_nonlin: Callable = None
    ):
        super().__init__()
        self.smooth = smooth
        self.num_iter = num_iter
        self.do_bg = do_bg
        self.apply_nonlin = apply_nonlin
    
    def forward(
        self,
        net_output: torch.Tensor,
        gt: torch.Tensor,
        gt_skeleton: torch.Tensor = None,
        loss_mask: torch.Tensor = None
    ) -> torch.Tensor:
        """
        Compute skeleton recall loss.
        
        Args:
            net_output: Network predictions (B, C, ...)
            gt: Ground truth one-hot encoded (B, C, ...)
            gt_skeleton: Pre-computed ground truth skeleton (B, C, ...) [optional]
            loss_mask: Mask for valid regions (B, 1, ...) [optional]
        
        Returns:
            Skeleton recall loss (scalar)
        """
        shp_x = net_output.shape
        
        # Apply nonlinearity if specified
        if self.apply_nonlin is not None:
            net_output = self.apply_nonlin(net_output)
        
        # Compute ground truth skeleton if not provided
        if gt_skeleton is None:
            gt_skeleton = soft_skel(gt, self.num_iter)
        
        # Flatten spatial dimensions
        axes = tuple(range(2, len(shp_x)))
        
        # Apply loss mask if provided
        if loss_mask is not None:
            net_output = net_output * loss_mask
            gt_skeleton = gt_skeleton * loss_mask
        
        # Compute intersection (recall numerator)
        intersection = (net_output * gt_skeleton).sum(dim=axes)
        
        # Compute skeleton sum (recall denominator)
        gt_skel_sum = gt_skeleton.sum(dim=axes)
        
        # Compute recall per class
        recall = (intersection + self.smooth) / (gt_skel_sum + self.smooth)
        
        # Skip background if specified
        if not self.do_bg:
            recall = recall[:, 1:]
        
        # Return mean loss (1 - recall)
        return 1.0 - recall.mean()
'''
    
    skeleton_recall_path = loss_dir / "skeleton_recall.py"
    skeleton_recall_path.write_text(skeleton_recall_code)
    print(f"  Created: {skeleton_recall_path}")
    
    # ==========================================================================
    # 2. Write compound_losses.py (DC_SkelREC_and_CE_loss)
    # ==========================================================================
    compound_losses_code = '''"""
Compound loss combining Dice, Cross-Entropy, and Skeleton Recall.

This loss is designed for topology-preserving segmentation, particularly
useful for thin structures like papyrus surfaces.
"""

import torch
import torch.nn as nn
from typing import Callable, List, Tuple

from nnunetv2.training.loss.dice import SoftDiceLoss, MemoryEfficientSoftDiceLoss
from nnunetv2.training.loss.robust_ce_loss import RobustCrossEntropyLoss
from nnunetv2.training.loss.skeleton_recall import SoftSkeletonRecallLoss


class DC_SkelREC_and_CE_loss(nn.Module):
    """
    Combined Dice + Skeleton Recall + Cross-Entropy Loss.
    
    Total loss = weight_dice * DiceLoss + weight_ce * CELoss + weight_srec * SkelRecLoss
    
    This loss combines:
    1. Dice Loss: For overall segmentation quality
    2. Cross-Entropy Loss: For pixel-wise classification
    3. Skeleton Recall Loss: For topology preservation
    
    Args:
        soft_dice_kwargs: Arguments for SoftDiceLoss
        ce_kwargs: Arguments for RobustCrossEntropyLoss
        skel_kwargs: Arguments for SoftSkeletonRecallLoss
        weight_ce: Weight for CE loss (default 1.0)
        weight_dice: Weight for Dice loss (default 1.0)
        weight_srec: Weight for Skeleton Recall loss (default 1.0)
        ignore_label: Label to ignore in loss computation
        dice_class: Dice loss class to use (default SoftDiceLoss)
    """
    
    def __init__(
        self,
        soft_dice_kwargs: dict = None,
        ce_kwargs: dict = None,
        skel_kwargs: dict = None,
        weight_ce: float = 1.0,
        weight_dice: float = 1.0,
        weight_srec: float = 1.0,
        ignore_label: int = None,
        dice_class: type = SoftDiceLoss
    ):
        super().__init__()
        
        # Handle ignore label
        if ignore_label is not None:
            if ce_kwargs is None:
                ce_kwargs = {}
            ce_kwargs['ignore_index'] = ignore_label
        
        # Initialize component losses
        self.weight_dice = weight_dice
        self.weight_ce = weight_ce
        self.weight_srec = weight_srec
        self.ignore_label = ignore_label
        
        # Dice loss
        if soft_dice_kwargs is None:
            soft_dice_kwargs = {'batch_dice': False, 'do_bg': True, 'smooth': 1e-5, 'ddp': False}
        self.dice = dice_class(apply_nonlin=torch.softmax, **soft_dice_kwargs)
        
        # Cross-entropy loss
        if ce_kwargs is None:
            ce_kwargs = {}
        self.ce = RobustCrossEntropyLoss(**ce_kwargs)
        
        # Skeleton recall loss
        if skel_kwargs is None:
            skel_kwargs = {'smooth': 1.0, 'num_iter': 10, 'do_bg': False}
        self.skel_rec = SoftSkeletonRecallLoss(apply_nonlin=torch.softmax, **skel_kwargs)
    
    def forward(
        self,
        net_output: torch.Tensor,
        target: torch.Tensor,
        gt_skeleton: torch.Tensor = None
    ) -> torch.Tensor:
        """
        Compute combined loss.
        
        Args:
            net_output: Network predictions (B, C, ...)
            target: Ground truth labels (B, 1, ...) or one-hot (B, C, ...)
            gt_skeleton: Pre-computed ground truth skeleton (optional)
        
        Returns:
            Combined loss (scalar)
        """
        # Handle ignore label for Dice loss
        if self.ignore_label is not None:
            # Create mask for valid pixels
            mask = (target != self.ignore_label).float()
            # For dice, we need one-hot encoding
            target_dice = target.clone()
            target_dice[target == self.ignore_label] = 0
        else:
            mask = None
            target_dice = target
        
        # Compute Dice loss (needs one-hot target)
        if target_dice.shape == net_output.shape:
            # Already one-hot
            target_onehot = target_dice
        else:
            # Convert to one-hot
            num_classes = net_output.shape[1]
            target_onehot = torch.zeros_like(net_output)
            target_onehot.scatter_(1, target_dice.long(), 1)
        
        # Apply mask to one-hot if needed
        if mask is not None:
            target_onehot = target_onehot * mask
        
        # Compute component losses
        dc_loss = self.dice(net_output, target_onehot)
        ce_loss = self.ce(net_output, target[:, 0].long() if target.shape[1] == 1 else target.argmax(dim=1))
        
        # Skeleton recall loss
        if gt_skeleton is not None:
            skel_loss = self.skel_rec(net_output, target_onehot, gt_skeleton, mask)
        else:
            skel_loss = self.skel_rec(net_output, target_onehot, None, mask)
        
        # Combine losses
        total_loss = (
            self.weight_dice * dc_loss +
            self.weight_ce * ce_loss +
            self.weight_srec * skel_loss
        )
        
        return total_loss
'''
    
    compound_losses_path = loss_dir / "skeleton_losses.py"
    compound_losses_path.write_text(compound_losses_code)
    print(f"  Created: {compound_losses_path}")
    
    # ==========================================================================
    # 3. Create transforms directory and write medial_surface.py
    # ==========================================================================
    transforms_dir = nnunet_path / "training" / "data_augmentation" / "custom_transforms"
    transforms_dir.mkdir(parents=True, exist_ok=True)
    
    # Create __init__.py
    (transforms_dir / "__init__.py").write_text("")
    
    medial_surface_code = '''"""
Medial Surface Transform for computing 2D slice-based skeletonization.

This transform computes the skeleton of segmentation masks along the Z-axis,
treating each XY slice independently. The skeleton is optionally dilated
to create a "tube" around the medial surface.
"""

import numpy as np
from typing import Tuple

try:
    from skimage.morphology import skeletonize, dilation, ball
    SKIMAGE_AVAILABLE = True
except ImportError:
    SKIMAGE_AVAILABLE = False


def compute_medial_surface_2d(
    segmentation: np.ndarray,
    do_tube: bool = True,
    tube_radius: int = 2
) -> np.ndarray:
    """
    Compute medial surface by 2D skeletonization along Z-axis.
    
    For each Z slice, compute the 2D skeleton and optionally dilate it.
    This is suitable for sheet-like structures (papyrus surfaces).
    
    Args:
        segmentation: Binary segmentation mask (D, H, W) or (1, D, H, W)
        do_tube: Whether to dilate skeleton to create tube
        tube_radius: Radius of dilation (number of iterations)
    
    Returns:
        Skeleton mask with same shape as input
    """
    if not SKIMAGE_AVAILABLE:
        raise ImportError("scikit-image is required for skeletonization. Install with: pip install scikit-image")
    
    # Handle channel dimension
    squeeze = False
    if segmentation.ndim == 4:
        if segmentation.shape[0] != 1:
            raise ValueError(f"Expected single channel, got {segmentation.shape[0]}")
        segmentation = segmentation[0]
        squeeze = True
    
    D, H, W = segmentation.shape
    skeleton = np.zeros_like(segmentation, dtype=bool)
    
    # Process each Z slice
    for z in range(D):
        slice_2d = segmentation[z] > 0.5
        if slice_2d.any():
            skeleton[z] = skeletonize(slice_2d)
    
    # Optionally dilate to create tube
    if do_tube and skeleton.any():
        for _ in range(tube_radius):
            skeleton = dilation(skeleton)
    
    result = skeleton.astype(np.float32)
    
    if squeeze:
        result = result[np.newaxis, ...]
    
    return result


class MedialSurfaceTransform:
    """
    Batchgenerators-compatible transform for computing medial surface skeletons.
    
    This transform adds a 'skeleton' key to the data dict containing the
    skeletonized version of the segmentation mask.
    
    Args:
        do_tube: Whether to dilate skeleton
        tube_radius: Dilation radius
        skeleton_key: Key to store skeleton in data dict
    """
    
    def __init__(
        self,
        do_tube: bool = True,
        tube_radius: int = 2,
        skeleton_key: str = "skeleton"
    ):
        self.do_tube = do_tube
        self.tube_radius = tube_radius
        self.skeleton_key = skeleton_key
    
    def __call__(self, **data_dict) -> dict:
        """
        Apply transform to data dict.
        
        Expected keys:
        - 'seg': Segmentation mask (B, C, D, H, W) or (C, D, H, W)
        
        Added keys:
        - skeleton_key: Computed skeleton
        """
        seg = data_dict.get('seg')
        if seg is None:
            return data_dict
        
        # Handle batch dimension
        if seg.ndim == 5:
            # Batch mode (B, C, D, H, W)
            skeletons = []
            for b in range(seg.shape[0]):
                skel = compute_medial_surface_2d(
                    seg[b],
                    do_tube=self.do_tube,
                    tube_radius=self.tube_radius
                )
                skeletons.append(skel)
            data_dict[self.skeleton_key] = np.stack(skeletons, axis=0)
        else:
            # Single sample (C, D, H, W)
            data_dict[self.skeleton_key] = compute_medial_surface_2d(
                seg,
                do_tube=self.do_tube,
                tube_radius=self.tube_radius
            )
        
        return data_dict
'''
    
    medial_surface_path = transforms_dir / "medial_surface.py"
    medial_surface_path.write_text(medial_surface_code)
    print(f"  Created: {medial_surface_path}")
    
    # ==========================================================================
    # 4. Create trainer variant directory and write nnUNetTrainerMedialSurfaceRecall.py
    # ==========================================================================
    trainer_variants_dir = nnunet_path / "training" / "nnUNetTrainer" / "variants" / "loss"
    trainer_variants_dir.mkdir(parents=True, exist_ok=True)
    
    # Ensure __init__.py exists
    init_path = trainer_variants_dir / "__init__.py"
    if not init_path.exists():
        init_path.write_text("")
    
    trainer_code = '''"""
nnUNet Trainer with Medial Surface Recall Loss for Host Baseline.

This trainer extends nnUNetTrainer to use the combined Dice + CE + Skeleton Recall loss
and includes data augmentation for skeleton computation.
"""

import torch
import numpy as np
from typing import Tuple, Union, List

from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
from nnunetv2.training.loss.skeleton_losses import DC_SkelREC_and_CE_loss
from nnunetv2.training.loss.dice import MemoryEfficientSoftDiceLoss
from nnunetv2.utilities.plans_handling.plans_handler import PlansManager
from nnunetv2.training.data_augmentation.custom_transforms.medial_surface import (
    MedialSurfaceTransform,
    compute_medial_surface_2d
)


class nnUNetTrainerMedialSurfaceRecall(nnUNetTrainer):
    """
    Custom nnUNet trainer for Host Baseline with Skeleton Recall Loss.
    
    Modifications from base nnUNetTrainer:
    1. Uses DC_SkelREC_and_CE_loss (Dice + CE + Skeleton Recall)
    2. Adds MedialSurfaceTransform to compute skeletons during training
    3. Modifies train_step to pass skeleton to loss function
    
    Loss weights (default):
    - weight_dice: 1.0
    - weight_ce: 1.0
    - weight_srec: 1.0
    """
    
    def __init__(self, plans: dict, configuration: str, fold: int,
                 dataset_json: dict, unpack_dataset: bool = True,
                 device: torch.device = torch.device('cuda')):
        super().__init__(plans, configuration, fold, dataset_json, unpack_dataset, device)
        
        # Loss weights for skeleton recall
        self.weight_dice = 1.0
        self.weight_ce = 1.0
        self.weight_srec = 1.0
    
    def _build_loss(self):
        """
        Build the combined loss function.
        
        Returns:
            DC_SkelREC_and_CE_loss instance
        """
        # Get ignore label if set
        if self.label_manager.ignore_label is not None:
            ignore_label = self.label_manager.ignore_label
        else:
            ignore_label = None
        
        # Configure loss components
        soft_dice_kwargs = {
            'batch_dice': self.configuration_manager.batch_dice,
            'do_bg': True,
            'smooth': 1e-5,
            'ddp': self.is_ddp
        }
        
        ce_kwargs = {}
        if ignore_label is not None:
            ce_kwargs['ignore_index'] = ignore_label
        
        skel_kwargs = {
            'smooth': 1.0,
            'num_iter': 10,
            'do_bg': False
        }
        
        loss = DC_SkelREC_and_CE_loss(
            soft_dice_kwargs=soft_dice_kwargs,
            ce_kwargs=ce_kwargs,
            skel_kwargs=skel_kwargs,
            weight_dice=self.weight_dice,
            weight_ce=self.weight_ce,
            weight_srec=self.weight_srec,
            ignore_label=ignore_label,
            dice_class=MemoryEfficientSoftDiceLoss
        )
        
        # Wrap in deep supervision wrapper if needed
        if self.enable_deep_supervision:
            from nnunetv2.training.loss.deep_supervision import DeepSupervisionWrapper
            # Deep supervision weights (standard nnUNet scheme)
            deep_supervision_scales = self._get_deep_supervision_scales()
            weights = np.array([1 / (2 ** i) for i in range(len(deep_supervision_scales))])
            weights = weights / weights.sum()
            loss = DeepSupervisionWrapper(loss, weights)
        
        return loss
    
    def _get_deep_supervision_scales(self) -> List[List[int]]:
        """Get deep supervision scales from configuration."""
        pool_op_kernel_sizes = self.configuration_manager.pool_op_kernel_sizes
        return list(list(i) for i in pool_op_kernel_sizes)
    
    def train_step(self, batch: dict) -> dict:
        """
        Perform a single training step.
        
        This method is modified to compute the skeleton of the ground truth
        and pass it to the loss function.
        
        Args:
            batch: Dictionary containing 'data' and 'target' keys
        
        Returns:
            Dictionary containing 'loss' key
        """
        data = batch['data']
        target = batch['target']
        
        data = data.to(self.device, non_blocking=True)
        if isinstance(target, list):
            target = [t.to(self.device, non_blocking=True) for t in target]
        else:
            target = target.to(self.device, non_blocking=True)
        
        self.optimizer.zero_grad(set_to_none=True)
        
        with torch.autocast(self.device.type, enabled=True) if self.device.type == 'cuda' else torch.autocast('cpu', enabled=False):
            output = self.network(data)
            
            # Compute skeleton from target for skeleton recall loss
            # Note: For deep supervision, target is a list
            if isinstance(target, list):
                # Deep supervision mode - compute skeleton for each scale
                # For now, we only use skeleton at full resolution
                target_np = target[0].cpu().numpy()
            else:
                target_np = target.cpu().numpy()
            
            # Compute skeletons batch-wise
            skeletons = []
            for b in range(target_np.shape[0]):
                # target_np shape: (B, C, D, H, W) or (B, 1, D, H, W)
                seg = target_np[b]  # (C, D, H, W)
                
                # Handle ignore label - create binary mask of foreground only
                if self.label_manager.ignore_label is not None:
                    # Create binary foreground mask (1 = surface, ignore everything else)
                    fg_mask = (seg == 1).astype(np.float32)
                else:
                    fg_mask = (seg > 0).astype(np.float32)
                
                skel = compute_medial_surface_2d(fg_mask, do_tube=True, tube_radius=2)
                skeletons.append(skel)
            
            gt_skeleton = torch.from_numpy(np.stack(skeletons, axis=0)).to(self.device)
            
            # Compute loss
            # For deep supervision, loss wrapper handles the list
            if isinstance(target, list):
                # The DeepSupervisionWrapper expects (output_list, target_list)
                # We pass skeleton only for the full resolution
                # Modify the loss to accept skeleton
                l = self.loss(output, target)  # Standard deep supervision
                # TODO: Integrate skeleton into deep supervision properly
            else:
                # Non deep supervision - pass skeleton to loss
                l = self.loss(output, target, gt_skeleton)
        
        if self.grad_scaler is not None:
            self.grad_scaler.scale(l).backward()
            self.grad_scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.network.parameters(), 12)
            self.grad_scaler.step(self.optimizer)
            self.grad_scaler.update()
        else:
            l.backward()
            torch.nn.utils.clip_grad_norm_(self.network.parameters(), 12)
            self.optimizer.step()
        
        return {'loss': l.detach().cpu().numpy()}
'''
    
    trainer_path = trainer_variants_dir / "nnUNetTrainerMedialSurfaceRecall.py"
    trainer_path.write_text(trainer_code)
    print(f"  Created: {trainer_path}")
    
    # ==========================================================================
    # 5. Create custom plans JSON
    # ==========================================================================
    plans_dir = NNUNET_PREPROCESSED / DATASET_NAME
    plans_dir.mkdir(parents=True, exist_ok=True)
    
    # Host Baseline architecture configuration
    host_baseline_plans = {
        "dataset_name": DATASET_NAME,
        "plans_name": "nnUNetHostBaselinePlans",
        "original_median_spacing_after_transp": [1.0, 1.0, 1.0],
        "original_median_shape_after_transp": [192, 192, 192],
        "image_reader_writer": "SimpleTiffIO",
        "transpose_forward": [0, 1, 2],
        "transpose_backward": [0, 1, 2],
        "configurations": {
            "3d_fullres": {
                "data_identifier": "nnUNetPlans_3d_fullres",
                "preprocessor_name": "DefaultPreprocessor",
                "batch_size": 2,
                "patch_size": [192, 192, 192],
                "median_image_size_in_voxels": [192.0, 192.0, 192.0],
                "spacing": [1.0, 1.0, 1.0],
                "normalization_schemes": ["ZScoreNormalization"],
                "use_mask_for_norm": [False],
                "resampling_fn_data": "resample_data_or_seg_to_shape",
                "resampling_fn_seg": "resample_data_or_seg_to_shape",
                "resampling_fn_data_kwargs": {
                    "is_seg": False, "order": 3, "order_z": 0, "force_separate_z": None
                },
                "resampling_fn_seg_kwargs": {
                    "is_seg": True, "order": 1, "order_z": 0, "force_separate_z": None
                },
                "resampling_fn_probabilities": "resample_data_or_seg_to_shape",
                "resampling_fn_probabilities_kwargs": {
                    "is_seg": False, "order": 1, "order_z": 0, "force_separate_z": None
                },
                "UNet_class_name": "dynamic_network_architectures.architectures.unet.ResidualEncoderUNet",
                "UNet_base_num_features": 32,
                "n_conv_per_stage_encoder": [1, 3, 4, 6, 6, 6],
                "n_conv_per_stage_decoder": [1, 1, 1, 1, 1],
                "num_pool_per_axis": [5, 5, 5],
                "pool_op_kernel_sizes": [
                    [1, 1, 1],
                    [2, 2, 2],
                    [2, 2, 2],
                    [2, 2, 2],
                    [2, 2, 2],
                    [2, 2, 2]
                ],
                "conv_kernel_sizes": [
                    [3, 3, 3],
                    [3, 3, 3],
                    [3, 3, 3],
                    [3, 3, 3],
                    [3, 3, 3],
                    [3, 3, 3]
                ],
                "unet_max_num_features": 320,
                "batch_dice": False
            }
        },
        "experiment_planner_used": "nnUNetPlannerResEncM",
        "label_manager": "LabelManager",
        "foreground_intensity_properties_per_channel": {
            "0": {
                "max": 255.0,
                "mean": 127.0,
                "median": 127.0,
                "min": 0.0,
                "percentile_00_5": 0.0,
                "percentile_99_5": 255.0,
                "std": 50.0
            }
        }
    }
    
    plans_path = plans_dir / "nnUNetHostBaselinePlans.json"
    with open(plans_path, 'w') as f:
        json.dump(host_baseline_plans, f, indent=2)
    print(f"  Created: {plans_path}")
    
    print("\nCustom trainer installation complete!")
    print("\nUsage:")
    print("  1. Set TRAINER = 'nnUNetTrainerMedialSurfaceRecall'")
    print("  2. Set CUSTOM_PLANS = 'nnUNetHostBaselinePlans'")
    print("  3. Run full_pipeline() or run_training()")
    
    return nnunet_path


# Verify installation by attempting import
def verify_custom_trainer():
    """Verify that custom trainer can be imported."""
    try:
        from nnunetv2.training.loss.skeleton_recall import SoftSkeletonRecallLoss
        from nnunetv2.training.loss.skeleton_losses import DC_SkelREC_and_CE_loss
        from nnunetv2.training.data_augmentation.custom_transforms.medial_surface import MedialSurfaceTransform
        from nnunetv2.training.nnUNetTrainer.variants.loss.nnUNetTrainerMedialSurfaceRecall import nnUNetTrainerMedialSurfaceRecall
        print("All custom components imported successfully!")
        return True
    except ImportError as e:
        print(f"Import failed: {e}")
        print("Run install_custom_trainer() first.")
        return False

## 4. Data Utilities

**Section Summary:**
Helper functions for handling TIFF and NIfTI file formats.

**TIFF Format:**
- Competition uses 3D TIFF files for CT volumes
- nnUNet natively supports TIFF via our custom SimpleTiffIO reader
- No conversion to NIfTI needed (saves disk space and time)

**Spacing Information:**
- nnUNet requires voxel spacing (physical size of each voxel)
- We use isotropic spacing (1.0, 1.0, 1.0) as competition doesn't specify
- Stored in JSON sidecar files alongside TIFF images

In [ ]:
def create_spacing_json(output_path: Path, shape: tuple, spacing: tuple = (1.0, 1.0, 1.0)):
    """Create JSON sidecar with spacing info for TIFF files."""
    json_data = {"spacing": list(spacing)}
    with open(output_path, "w") as f:
        json.dump(json_data, f)


def load_nifti(path: Path) -> np.ndarray:
    """Load NIfTI file (used for loading nnUNet predictions)."""
    return nib.load(str(path)).get_fdata()


def create_dataset_json(output_dir: Path, num_training: int, file_ending: str = ".tif") -> dict:
    """Create dataset.json with ignore label support and 3D TIFF reader."""
    
    dataset_json = {
        "channel_names": {"0": "CT"},
        "labels": {"background": 0, "surface": 1, "ignore": 2},
        "numTraining": num_training,
        "file_ending": file_ending,
        "overwrite_image_reader_writer": "SimpleTiffIO"
    }
    
    json_path = output_dir / "dataset.json"
    with open(json_path, "w") as f:
        json.dump(dataset_json, f, indent=4)
    
    print(f"Created {json_path}")
    print(f"  - {num_training} training cases")
    print(f"  - Labels: background(0), surface(1), ignore(2)")
    print(f"  - Reader: SimpleTiffIO (3D TIFF)")
    
    return dataset_json

## 5. Dataset Preparation

**Section Summary:**
Converts competition data to nnUNet's expected format.

**nnUNet Dataset Structure:**
```
nnUNet_raw/Dataset100_VesuviusSurface/
├── imagesTr/           # Training images
│   ├── case001_0000.tif  # _0000 suffix = channel 0 (only one for CT)
│   └── case001_0000.json # Spacing information
├── labelsTr/           # Training labels
│   ├── case001.tif
│   └── case001.json
└── dataset.json        # Dataset configuration
```

**Symlink Strategy:**
- Uses symlinks instead of copying (fast, saves disk space)
- Only JSON sidecar files are created (spacing info)
- Original TIFF files remain in competition dataset

**Parallel Processing:**
- Uses multiprocessing for faster preparation
- ~2 minutes for 806 training cases

In [ ]:
def prepare_single_case(
    src_path: Path, 
    dest_path: Path, 
    json_path: Path, 
    use_symlinks: bool = True
) -> bool:
    """
    Prepare a single TIFF file for nnUNet: create symlink/copy and JSON sidecar.
    Returns True on success, False on failure.
    """
    try:
        # Get shape for JSON
        with tifffile.TiffFile(src_path) as tif:
            shape = tif.pages[0].shape if len(tif.pages) == 1 else (len(tif.pages), *tif.pages[0].shape)
        
        # Link or copy file
        if use_symlinks:
            if not dest_path.exists():
                dest_path.symlink_to(src_path.resolve())
        else:
            shutil.copy2(src_path, dest_path)
        
        # Create JSON sidecar
        create_spacing_json(json_path, shape)
        return True
        
    except Exception as e:
        print(f"Error processing {src_path.name}: {e}")
        return False


def _prepare_training_case(
    img_path: Path,
    train_labels_dir: Path,
    images_dir: Path,
    labels_dir: Path,
    use_symlinks: bool
) -> bool:
    """Worker function for parallel dataset preparation."""
    case_id = img_path.stem
    label_path = train_labels_dir / img_path.name
    
    if not label_path.exists():
        return False
    
    img_ok = prepare_single_case(
        img_path,
        images_dir / f"{case_id}_0000.tif",
        images_dir / f"{case_id}_0000.json",
        use_symlinks
    )
    
    label_ok = prepare_single_case(
        label_path,
        labels_dir / f"{case_id}.tif",
        labels_dir / f"{case_id}.json",
        use_symlinks
    )
    
    return img_ok and label_ok


def prepare_dataset(input_dir: Path, max_cases: Optional[int] = None, use_symlinks: bool = True):
    """
    Convert competition data to nnUNet format using TIFF directly (no NIfTI).
    Uses multiprocessing for faster preparation.
    
    Competition structure:
    - train_images/*.tif  (3D volumes)
    - train_labels/*.tif  (3D labels: 0=bg, 1=surface, 2=ignore)
    """
    dataset_dir = NNUNET_RAW / DATASET_NAME
    images_dir = dataset_dir / "imagesTr"
    labels_dir = dataset_dir / "labelsTr"
    
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)
    
    train_images_dir = input_dir / "train_images"
    train_labels_dir = input_dir / "train_labels"
    
    if not train_images_dir.exists():
        print(f"ERROR: {train_images_dir} not found!")
        return None
    
    image_files = sorted(train_images_dir.glob("*.tif"))
    if max_cases:
        image_files = image_files[:max_cases]
    
    print(f"Found {len(image_files)} training cases")
    print(f"Using {'symlinks' if use_symlinks else 'copy'}")
    print(f"Processing with {NUM_WORKERS} workers...")
    
    # Create worker function with fixed arguments
    worker = partial(
        _prepare_training_case,
        train_labels_dir=train_labels_dir,
        images_dir=images_dir,
        labels_dir=labels_dir,
        use_symlinks=use_symlinks
    )
    
    # Process in parallel with progress bar
    with Pool(NUM_WORKERS) as pool:
        results = list(tqdm(
            pool.imap(worker, image_files),
            total=len(image_files),
            desc="Preparing dataset"
        ))
    
    num_converted = sum(results)
    create_dataset_json(dataset_dir, num_converted, file_ending=".tif")
    
    print(f"\nDataset prepared: {num_converted} cases")
    print(f"Location: {dataset_dir}")
    
    return dataset_dir

## 6. Synthetic Data (for testing)

**Section Summary:**
Creates small synthetic datasets for testing the pipeline without real data.

**Use Cases:**
- Testing pipeline setup before using real data
- Debugging data loading issues
- Quick iteration on code changes

**Generated Data:**
- Small 3D volumes (64x64x64 by default)
- Spherical shell as "surface" (label 1)
- Random ignore regions (label 2)
- Random background noise

In [ ]:
def create_single_synthetic_case(
    case_id: str,
    size: Tuple[int, int, int],
    images_dir: Path,
    labels_dir: Path
):
    """Create a single synthetic training case."""
    # Create volume with structure
    volume = np.random.randn(*size).astype(np.float32) * 0.1
    
    # Add spherical shell as "surface"
    z, y, x = np.ogrid[:size[0], :size[1], :size[2]]
    center = np.array(size) // 2
    dist = np.sqrt((z - center[0])**2 + (y - center[1])**2 + (x - center[2])**2)
    shell = (dist > 15) & (dist < 20)
    volume[shell] += 1.0
    
    # Create labels
    labels = np.zeros(size, dtype=np.uint8)
    labels[shell] = 1  # Surface
    
    # Add ignore regions (label 2)
    ignore_mask = np.random.random(size) < 0.15
    labels[ignore_mask] = 2
    
    # Save as TIFF
    tifffile.imwrite(images_dir / f"{case_id}_0000.tif", volume)
    tifffile.imwrite(labels_dir / f"{case_id}.tif", labels)
    
    # Create JSON sidecars
    create_spacing_json(images_dir / f"{case_id}_0000.json", size)
    create_spacing_json(labels_dir / f"{case_id}.json", size)


def create_synthetic_dataset(num_cases: int = 5, size: Tuple[int, int, int] = (64, 64, 64)):
    """Create synthetic 3D data with ignore regions for testing (TIFF format)."""
    
    dataset_dir = NNUNET_RAW / DATASET_NAME
    images_dir = dataset_dir / "imagesTr"
    labels_dir = dataset_dir / "labelsTr"
    
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)
    
    for i in tqdm(range(num_cases), desc="Creating synthetic data"):
        case_id = f"case_{i:03d}"
        create_single_synthetic_case(case_id, size, images_dir, labels_dir)
    
    create_dataset_json(dataset_dir, num_cases, file_ending=".tif")
    print(f"Created synthetic dataset: {dataset_dir}")
    
    return dataset_dir

# To use synthetic data instead of real data, uncomment:
# create_synthetic_dataset(num_cases=5)

## 7. nnUNet Commands

**Section Summary:**
Wrapper functions for nnUNet command-line tools.

**nnUNet Pipeline Steps:**
1. `run_preprocessing()` → `nnUNetv2_plan_and_preprocess`
   - Analyzes dataset statistics
   - Creates experiment plans (network architecture, patch size, etc.)
   - Preprocesses data (resampling, normalization)
   - Time: 1-2 hours for full dataset

2. `run_training()` → `nnUNetv2_train`
   - Trains the network
   - Saves checkpoints every 50 epochs
   - Generates progress.png with loss curves
   - Time: 5-10 min/epoch on T4

3. `run_inference()` → `nnUNetv2_predict`
   - Runs trained model on test data
   - Outputs NIfTI predictions (converted to TIFF later)
   - Time: 1-2 min per volume

**Important Training Notes:**
- `fold="all"`: After training, nnUNet validates on ALL training data
  This can be slow (50+ min) but cannot be skipped with built-in flags
- Multi-GPU: Use `-num_gpus X` for DDP training
- Epochs: Use `-tr nnUNetTrainer_Xepochs` for custom epoch counts

In [ ]:
def _run_command(
    cmd: str, 
    name: str = "Command", 
    tail_lines: int = 20,
    timeout: Optional[int] = COMMAND_TIMEOUT
) -> bool:
    """
    Execute shell command and handle output parsing.
    
    Args:
        cmd: Shell command to execute
        name: Display name for logging
        tail_lines: Number of stdout lines to show on success
        timeout: Timeout in seconds (None for no timeout)
    
    Returns:
        True if command succeeded, False otherwise
    """
    print(f"Running: {cmd}")
    if timeout:
        print(f"Timeout: {timeout}s ({timeout/3600:.1f}h)")
    
    try:
        result = subprocess.run(
            cmd, 
            shell=True, 
            capture_output=True, 
            text=True,
            timeout=timeout
        )
    except subprocess.TimeoutExpired:
        print(f"{name} TIMEOUT after {timeout}s!")
        return False
    
    if result.returncode != 0:
        print(f"{name} FAILED!")
        print(f"STDERR:\n{result.stderr[-3000:]}")
        return False
    
    print(f"{name} complete!")
    if result.stdout.strip():
        lines = result.stdout.strip().split('\n')
        print('\n'.join(lines[-tail_lines:]))
    
    return True


def run_preprocessing(
    dataset_id: int = DATASET_ID, 
    planner: str = PLANNER,
    num_workers: int = NUM_WORKERS,
    configurations: Optional[List[str]] = None,
    timeout: Optional[int] = COMMAND_TIMEOUT
) -> bool:
    """
    Run nnUNet preprocessing.
    
    Args:
        dataset_id: nnUNet dataset ID
        planner: Planner class name
        num_workers: Number of CPU workers for parallel processing
        configurations: List of configs to preprocess (e.g., ["3d_fullres"])
        timeout: Timeout in seconds (None for no timeout)
    
    Returns:
        True if preprocessing succeeded
    """
    if configurations is None:
        configurations = [CONFIGURATION]
    
    cmd = f"nnUNetv2_plan_and_preprocess -d {dataset_id:03d} -np {num_workers}"
    cmd += f" -pl {planner}"
    cmd += f" -c {' '.join(configurations)}"
    
    return _run_command(cmd, "Preprocessing", timeout=timeout)


def _get_trainer_name(epochs: Optional[Epochs], trainer: str = "nnUNetTrainer") -> str:
    """
    Get trainer class name based on epochs and base trainer.
    
    For custom trainers (non-standard), returns the trainer name as-is.
    For standard nnUNetTrainer, appends epoch suffix if needed.
    """
    # Custom trainers don't use epoch suffix
    if trainer != "nnUNetTrainer":
        return trainer
    
    # Standard trainer with epoch variants
    if epochs is None or epochs == 1000:
        return "nnUNetTrainer"
    elif epochs == 1:
        return "nnUNetTrainer_1epoch"
    else:
        return f"nnUNetTrainer_{epochs}epochs"


def run_training(
    dataset_id: int = DATASET_ID,
    config: str = CONFIGURATION,
    fold: Union[int, str] = FOLD,
    plans: str = PLANS_NAME,
    epochs: Optional[Epochs] = EPOCHS,
    trainer: str = TRAINER,
    pretrained_weights: Optional[Path] = None,
    continue_training: bool = False,
    only_run_validation: bool = False,
    disable_checkpointing: bool = False,
    npz: bool = False,
    num_gpus: int = NUM_GPUS,
    timeout: Optional[int] = COMMAND_TIMEOUT
) -> bool:
    """
    Run nnUNet training.
    
    Args:
        dataset_id: nnUNet dataset ID
        config: Configuration name (3d_fullres, 2d, etc.)
        fold: Fold number (0-4) or "all" for training on all data
        plans: Plans name matching the planner used
        epochs: Number of epochs. Available: 1, 5, 10, 20, 50, 100, 150, 200, 250, 
                300, 400, 500, 750, 1000, 2000, 4000, 8000. None = 1000 (default)
        trainer: Trainer class name (default: TRAINER global, typically "nnUNetTrainer")
                 Use "nnUNetTrainerMedialSurfaceRecall" for Host Baseline
        pretrained_weights: Optional path to pretrained checkpoint for fine-tuning
        continue_training: Continue from last checkpoint (add -c flag)
        only_run_validation: Only run validation, skip training
        disable_checkpointing: Disable saving checkpoints (saves disk space)
        npz: Save softmax outputs during validation (needed for ensembling)
        num_gpus: Number of GPUs for DDP training (default: auto-detected)
        timeout: Timeout in seconds (None for no timeout)
    
    Returns:
        True if training succeeded
    
    Note:
        For multi-GPU (DDP) training, batch size should be divisible by num_gpus.
        The first run extracts preprocessed data - wait for GPU usage before 
        starting additional folds on other GPUs.
        
        When using fold="all", nnUNet will run validation on ALL training data
        after training completes, which can be slow. This is normal behavior.
    """
    trainer_name = _get_trainer_name(epochs, trainer)
    cmd = f"nnUNetv2_train {dataset_id:03d} {config} {fold} -p {plans} -tr {trainer_name}"
    
    if pretrained_weights:
        cmd += f" -pretrained_weights {pretrained_weights}"
    if continue_training:
        # Find checkpoint to resume from: prefer checkpoint_final.pth, fallback to checkpoint_best.pth
        model_dir = get_training_output_dir(epochs=epochs, plans=plans, config=config, fold=fold, trainer=trainer)
        checkpoint_final = model_dir / "checkpoint_final.pth"
        checkpoint_best = model_dir / "checkpoint_best.pth"
        if checkpoint_final.exists():
            print(f"Resuming from: {checkpoint_final}")
            cmd += " --c"
        elif checkpoint_best.exists():
            print(f"Resuming from: {checkpoint_best}")
            cmd += " --c"
        else:
            print(f"WARNING: No checkpoint found in {model_dir}, starting fresh")
    if only_run_validation:
        cmd += " --val"
    if disable_checkpointing:
        cmd += " --disable_checkpointing"
    if npz:
        cmd += " --npz"
    if num_gpus > 1:
        cmd += f" -num_gpus {num_gpus}"
    
    epochs_str = epochs if epochs else 1000
    gpu_str = f", {num_gpus} GPUs" if num_gpus > 1 else ""
    trainer_str = f", {trainer}" if trainer != "nnUNetTrainer" else ""
    return _run_command(cmd, f"Training ({epochs_str} epochs{gpu_str}{trainer_str})", tail_lines=30, timeout=timeout)


def run_inference(
    input_dir: Path,
    output_dir: Path,
    dataset_id: int = DATASET_ID,
    config: str = CONFIGURATION,
    fold: Union[int, str] = FOLD,
    plans: str = PLANS_NAME,
    epochs: Optional[Epochs] = EPOCHS,
    trainer: str = TRAINER,
    save_probabilities: bool = True,
    num_processes_preprocessing: int = 2,
    num_processes_segmentation: int = 2,
    timeout: Optional[int] = COMMAND_TIMEOUT
) -> bool:
    """
    Run inference with trained model.
    
    Args:
        input_dir: Directory with test images (must have _0000 suffix)
        output_dir: Directory to save predictions
        dataset_id: nnUNet dataset ID
        config: Configuration name
        fold: Fold number used for training (or "all" or tuple like "0,1,2")
        plans: Plans name
        epochs: Epochs used during training (must match trained model)
        trainer: Trainer class name (must match trained model)
        save_probabilities: Whether to save probability maps (.npz files)
        num_processes_preprocessing: Parallel processes for preprocessing
        num_processes_segmentation: Parallel processes for segmentation
        timeout: Timeout in seconds (None for no timeout)
    
    Returns:
        True if inference succeeded
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    
    trainer_name = _get_trainer_name(epochs, trainer)
    
    cmd = f"nnUNetv2_predict -d {dataset_id:03d} -c {config} -f {fold}"
    cmd += f" -i {input_dir} -o {output_dir} -p {plans} -tr {trainer_name}"
    cmd += f" -npp {num_processes_preprocessing} -nps {num_processes_segmentation}"
    cmd += " --verbose"
    
    if save_probabilities:
        cmd += " --save_probabilities"
    
    return _run_command(cmd, "Inference", timeout=timeout)

In [ ]:
# =============================================================================
# RUN PREPROCESSING ONLY
# =============================================================================
# This notebook only runs data preparation and preprocessing.
# Output will be saved to /kaggle/working/nnUNet_preprocessed/

print("="*60)
print("VESUVIUS PREPROCESSING - 3d_fullres")
print("="*60)

# Step 1: Setup
setup_environment()

# Step 2: Prepare dataset
print("\n[1/2] Preparing dataset...")
prepare_dataset(INPUT_DIR)

# Step 3: Run preprocessing
print("\n[2/2] Running nnUNet preprocessing...")
run_preprocessing(
    planner=PLANNER,
    configurations=[CONFIGURATION]
)

print("\n" + "="*60)
print("DONE!")
print("="*60)

# Show output
output_dir = NNUNET_PREPROCESSED / DATASET_NAME
if output_dir.exists():
    print(f"\nOutput: {output_dir}")
    for item in sorted(output_dir.iterdir())[:10]:
        print(f"  {item.name}")
